In [ ]:
pip install databento pandas numpy matplotlib


In [ ]:
import databento as db
import pandas as pd

# 1. Load just ONE day of data
path = "data/raw/xnas-itch-20230801.mbp-1.dbn.zst"
store = db.DBNStore.from_file(path)

# 2. Convert to a Pandas DataFrame
# Note: MBP-1 data includes both trades and quote updates
df = store.to_df()

# 3. Look at the columns
print(df.columns)
print(df.head())

In [ ]:
import numpy as np

# 1. Filter for TRADES only (Action 'T')
# In MBP-1, 'T' indicates an execution (a trade)
trades = df[df['action'] == 'T'].copy()

# 2. Filter for Regular Trading Hours (09:30 - 16:00)
trades = trades.between_time('09:30', '16:00')

# 3. Calculate the Midpoint from the prevailing Best Bid/Offer
trades['midpoint'] = (trades['bid_px_00'] + trades['ask_px_00']) / 2

# 4. Apply the 'Quote Rule' (Lee-Ready Step 1)
trades['side_lr'] = 0  # Initialize a new column
trades.loc[trades['price'] > trades['midpoint'], 'side_lr'] = 1  # Buy
trades.loc[trades['price'] < trades['midpoint'], 'side_lr'] = -1 # Sell

# 5. Apply the 'Tick Rule' (Lee-Ready Step 2 for midpoint trades)
# If price is exactly at midpoint, look at the price change from the previous trade
trades['price_diff'] = trades['price'].diff()
midpoint_trades = (trades['side_lr'] == 0)

trades.loc[midpoint_trades & (trades['price_diff'] > 0), 'side_lr'] = 1
trades.loc[midpoint_trades & (trades['price_diff'] < 0), 'side_lr'] = -1

# 6. Recursive Rule (If price didn't move, use the previous classification)
trades['side_lr'] = trades['side_lr'].replace(0, np.nan).ffill()

# 7. Calculate Signed Volume (This is a key input for Phase 2)
trades['signed_vol'] = trades['side_lr'] * trades['size']

print(f"Processed {len(trades)} trades.")
display(trades[['symbol', 'price', 'midpoint', 'side_lr', 'signed_vol']].head(10))